# 06 — Create AI/BI Dashboard (Lakeview API)

Creates and publishes an AI/BI (Lakeview) dashboard programmatically via the
Databricks REST API. The dashboard covers:

1. **Deal Volume by Conference** — bar chart
2. **Top 10 Athletes by Total Deal Value** — table
3. **Sponsor Spend by Industry** — bar chart
4. **Campaign Performance by Platform** — bar chart
5. **Monthly Deal Trend** — line chart
6. **KPI Summary** — counter widgets

The dashboard is then published so it appears in Databricks One and can be
added to a Domain on the Discover page.

In [1]:
from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv
import os, json

load_dotenv()

w = WorkspaceClient()

UC_CATALOG       = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA        = os.getenv("UC_SCHEMA", "nil_sponsorships")
GOLD_SCHEMA      = f"{UC_SCHEMA}_gold"
SQL_WAREHOUSE_ID = os.getenv("SQL_WAREHOUSE_ID")

if not SQL_WAREHOUSE_ID:
    raise ValueError("SQL_WAREHOUSE_ID not set in .env — find it under SQL Warehouses > Connection details")

G = f"{UC_CATALOG}.{GOLD_SCHEMA}"
print(f"Gold schema: {G}")
print(f"Warehouse:   {SQL_WAREHOUSE_ID}")

Gold schema: alexander_booth.nil_sponsorships_gold
Warehouse:   49458fa9801b109b


In [2]:
DASHBOARD_NAME = "NIL Sponsorship Analytics"

# Define datasets (queries the dashboard will use)
datasets = [
    {
        "name": "conference_deals",
        "displayName": "Deal Volume by Conference",
        "query": f"SELECT * FROM {G}.v_conference_summary ORDER BY total_deal_value DESC"
    },
    {
        "name": "athlete_leaderboard",
        "displayName": "Top Athletes by Deal Value",
        "query": f"SELECT * FROM {G}.v_athlete_deal_leaderboard ORDER BY total_deal_value DESC LIMIT 10"
    },
    {
        "name": "sponsor_roi",
        "displayName": "Sponsor ROI by Industry",
        "query": f"SELECT industry, SUM(total_invested) AS total_invested, SUM(total_impressions) AS total_impressions, SUM(total_conversions) AS total_conversions FROM {G}.v_sponsor_roi GROUP BY industry ORDER BY total_invested DESC"
    },
    {
        "name": "campaign_by_platform",
        "displayName": "Campaign Performance by Platform",
        "query": f"SELECT platform, SUM(impressions) AS total_impressions, SUM(engagements) AS total_engagements, SUM(clicks) AS total_clicks, SUM(conversions) AS total_conversions, ROUND(AVG(engagement_rate), 4) AS avg_engagement_rate FROM {G}.fact_campaign_performance GROUP BY platform ORDER BY total_impressions DESC"
    },
    {
        "name": "monthly_deals",
        "displayName": "Monthly Deal Trend",
        "query": f"SELECT DATE_TRUNC('month', start_date) AS deal_month, COUNT(*) AS deal_count, SUM(deal_amount) AS total_value FROM {G}.fact_deals WHERE status != 'Cancelled' GROUP BY 1 ORDER BY 1"
    },
    {
        "name": "kpi_summary",
        "displayName": "KPI Summary",
        "query": f"SELECT COUNT(DISTINCT deal_id) AS total_deals, SUM(deal_amount) AS total_deal_value, COUNT(DISTINCT athlete_sk_fk) AS unique_athletes, COUNT(DISTINCT sponsor_sk_fk) AS unique_sponsors FROM {G}.fact_deals WHERE status IN ('Active', 'Completed')"
    }
]

# Build a minimal dashboard JSON with just datasets and an empty page
dashboard_json = {
    "pages": [
        {
            "name": "nil_overview",
            "displayName": "NIL Sponsorship Overview"
        }
    ],
    "datasets": datasets
}

serialized = json.dumps(dashboard_json)
print(f"Dashboard JSON: {len(serialized):,} chars")
print(f"Datasets: {len(datasets)}")
for ds in datasets:
    print(f"  - {ds['displayName']}")

Dashboard JSON: 1,889 chars
Datasets: 6
  - Deal Volume by Conference
  - Top Athletes by Deal Value
  - Sponsor ROI by Industry
  - Campaign Performance by Platform
  - Monthly Deal Trend
  - KPI Summary


In [3]:
# Find existing dashboard or create a new one
from databricks.sdk.service.dashboards import Dashboard

current_user = w.current_user.me()
parent_path = f"/Workspace/Users/{current_user.user_name}"

# Check if a dashboard with this name already exists by looking for the file
existing_id = None
try:
    for obj in w.workspace.list(parent_path):
        if obj.path and obj.path.endswith(f"{DASHBOARD_NAME}.lvdash.json"):
            # Get the dashboard ID from the object
            existing_id = str(obj.object_id)
            break
except Exception:
    pass

if existing_id:
    # Fetch the actual dashboard to get its dashboard_id
    try:
        dashboard = w.lakeview.get(dashboard_id=existing_id)
        dashboard_id = dashboard.dashboard_id
        w.lakeview.update(
            dashboard_id=dashboard_id,
            dashboard=Dashboard(serialized_dashboard=serialized, warehouse_id=SQL_WAREHOUSE_ID),
        )
        print(f"Dashboard already exists — updated: {dashboard_id}")
    except Exception:
        existing_id = None

if not existing_id:
    dashboard = w.lakeview.create(
        dashboard=Dashboard(
            display_name=DASHBOARD_NAME,
            warehouse_id=SQL_WAREHOUSE_ID,
            serialized_dashboard=serialized,
            parent_path=parent_path,
        )
    )
    dashboard_id = dashboard.dashboard_id
    print(f"Dashboard created: {dashboard_id}")

print(f"Name: {DASHBOARD_NAME}")

Dashboard already exists — updated: 01f13297cbad133a9fb9d7b73fe89a87
Name: NIL Sponsorship Analytics


In [4]:
# Publish the dashboard so it's visible to consumers in Databricks One
w.lakeview.publish(
    dashboard_id=dashboard_id,
    warehouse_id=SQL_WAREHOUSE_ID,
    embed_credentials=True,
)

host = os.getenv("DATABRICKS_HOST", "").rstrip("/")
print(f"Dashboard published!")
print(f"\nView at: {host}/sql/dashboardsv3/{dashboard_id}")
print(f"\nThis dashboard is now visible in Databricks One and can be added to a Domain.")

Dashboard published!

View at: https://e2-demo-field-eng.cloud.databricks.com/sql/dashboardsv3/01f13297cbad133a9fb9d7b73fe89a87

This dashboard is now visible in Databricks One and can be added to a Domain.
